### bulkformer-latest feature extraction

This example uses four **synthetic count profiles** and the released Stage1 epoch 17 checkpoint (19,973 genes; 768-dimensional embeddings).

Create the environment using `configs/bulkformer-latest.yaml` and download `model.pt`, `model_config.json`, `gene_vocab.csv`, `edge_index.pt`, `edge_weight.pt`, and `manifest.json` from the [Google Drive folder](https://drive.google.com/drive/folders/1AfekiOBNs6GzPBbA9BDlpuoolZIWbGQ2) into `model/bulkformer-latest/`. Run this notebook inside the cloned repository. See the [model card](../docs/models/bulkformer-latest.md) for details.


In [1]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")


'7'

In [2]:
from pathlib import Path
import sys

# Use repository-relative paths when opening the notebook from notebooks/.
if Path.cwd().name == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())


In [3]:
import pandas as pd
import numpy as np
import torch
from utils.latest import prepare_latest_bundle, load_latest_model
from utils.latest import normalize_data, main_gene_selection, extract_feature


In [4]:
# Configuration
device = 'cuda'
batch_size = 1
amp_dtype = 'bf16' if torch.cuda.is_bf16_supported() else 'fp16'
model_path = 'model/bulkformer-latest'
data_path = 'data/latest_demo_counts.csv'
output_path = 'outputs/latest_demo'


In [5]:
torch.cuda.get_device_name(0), amp_dtype


('NVIDIA A800 80GB PCIe', 'bf16')

In [6]:
# Load the pretrained model and its matching vocabulary and graph.
model_dir = prepare_latest_bundle(model_path)
model, bulkformer_gene_info, manifest = load_latest_model(model_dir, device=device)


In [7]:
manifest['checkpoint_metadata']


{'stage': 'Stage1',
 'epoch': 17,
 'global_update_step': 126055,
 'saved_at': '2026-10-01T08:32:05+08:00',
 'best_metric': 0.1891963980159657,
 'selection_metric': 'mean ID/OOD masked-expression MSE at 10% masking'}

In [8]:
bulkformer_gene_info.head()


,vocab_index,canonical_ensg,gene_symbol,gene_length_bp
0,0,ENSG00000000003,TSPAN6,4536.0
1,1,ENSG00000000005,TNMD,1476.0
2,2,ENSG00000000419,DPM1,9276.0
3,3,ENSG00000000457,SCYL3,6883.0
4,4,ENSG00000000460,C1orf112,5970.0


In [9]:
# Load demo count data (rows: samples; columns: Ensembl gene IDs).
count_df = pd.read_csv(data_path, index_col=0)


In [10]:
count_df.shape


(4, 19965)

In [11]:
count_df.iloc[:, :6]


,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938
synthetic_1,0,19,42,32,44,53
synthetic_2,0,41,37,48,16,25
synthetic_3,0,53,66,11,4,36
synthetic_4,0,32,9,17,56,14


In [12]:
# Convert raw counts to ln(TPM+1) using the released gene lengths.
log_tpm_df = normalize_data(X_df=count_df, gene_vocab=bulkformer_gene_info)

# For input already in ln(TPM+1), load it directly instead of normalizing counts.
# log_tpm_df = pd.read_csv('data/your_log_tpm.csv', index_col=0)


In [13]:
log_tpm_df.iloc[:, :6]


,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938
synthetic_1,0.0,4.251817,3.232856,3.258268,3.704731,4.445992
synthetic_2,0.0,5.005400,3.103841,3.643105,2.727873,3.699885
synthetic_3,0.0,5.273841,3.675632,2.266088,1.531038,4.070024
synthetic_4,0.0,4.784697,1.842226,2.675428,3.957799,3.163684


In [14]:
bulkformer_gene_list = bulkformer_gene_info['canonical_ensg'].to_list()


In [15]:
# Align the complete gene axis. Missing values become -10; measured zeros stay 0.
input_df, to_fill_columns, var = main_gene_selection(
    X_df=log_tpm_df, gene_list=bulkformer_gene_list)


In [16]:
input_df.shape


(4, 19973)

In [17]:
pd.DataFrame({
    'missing_genes': (input_df == -10).sum(axis=1),
    'missing_rate': (input_df == -10).mean(axis=1),
    'measured_zeros': (input_df == 0).sum(axis=1)
})


,missing_genes,missing_rate,measured_zeros
synthetic_1,8,0.000401,1197
synthetic_2,8,0.000401,1190
synthetic_3,8,0.000401,1186
synthetic_4,8,0.000401,1194


In [18]:
# Choose genes for gene-level output; replace this list with your Ensembl IDs.
interested_gene_list = bulkformer_gene_list[:8]
interested_gene_idx = [bulkformer_gene_list.index(gene) for gene in interested_gene_list]


In [19]:
bulkformer_gene_info.iloc[interested_gene_idx][['canonical_ensg', 'gene_symbol']]


,canonical_ensg,gene_symbol
0,ENSG00000000003,TSPAN6
1,ENSG00000000005,TNMD
2,ENSG00000000419,DPM1
3,ENSG00000000457,SCYL3
4,ENSG00000000460,C1orf112
5,ENSG00000000938,FGR
6,ENSG00000000971,CFH
7,ENSG00000001036,FUCA2


In [20]:
# Extract sample-level embedding (mean over observed genes, including zeros).
res1 = extract_feature(
    model=model,
    expr_array=input_df.values,
    output_feature_type='sample_level',
    aggregate_type='mean',
    batch_size=batch_size,
    amp_dtype=amp_dtype)


In [21]:
res1.shape


(4, 768)

In [22]:
res1


array([[ 0.05447929,  0.00416262, -0.01259869, ...,  0.2624566 ,
        -0.22775164, -0.15894453],
       [ 0.04759149,  0.00654286, -0.00899188, ...,  0.23926784,
        -0.22674198, -0.14086565],
       [ 0.06186718,  0.01633062, -0.02844604, ...,  0.27487776,
        -0.23684402, -0.11808996],
       [ 0.09571672, -0.00109521, -0.01814602, ...,  0.266644  ,
        -0.23167133, -0.13327639]], dtype=float32)

In [23]:
# Extract contextual gene-level embedding for the selected genes.
res2 = extract_feature(
    model=model,
    expr_array=input_df.values,
    output_feature_type='gene_level',
    interested_gene_idx=interested_gene_idx,
    batch_size=batch_size,
    amp_dtype=amp_dtype)


In [24]:
res2.shape


(4, 8, 768)

In [25]:
res2


array([[[-2.86134154e-01,  1.11627114e+00,  1.48499966e-01, ...,
          5.60762882e-01, -1.16620377e-01,  2.72716999e-01],
        [ 1.67810500e-01, -7.31194556e-01, -1.29959688e-01, ...,
          2.52438158e-01,  4.46575403e-01, -2.74839669e-01],
        [-2.31271595e-01,  2.79058274e-02, -4.59142834e-01, ...,
          1.57252967e-01, -3.28603894e-01, -1.93622440e-01],
        ...,
        [ 5.25801837e-01,  1.42292786e+00, -7.29236722e-01, ...,
         -2.33931188e-02, -4.79897745e-02, -5.11137128e-01],
        [ 9.45439115e-02, -6.03495717e-01, -8.03443491e-02, ...,
          1.95878297e-01,  4.26548928e-01, -2.50479013e-01],
        [ 2.57988453e-01, -8.67802024e-01, -6.22834384e-01, ...,
          5.85035920e-01,  5.49922407e-01,  3.09846103e-01]],

       [[-2.86914527e-01,  1.13198411e+00,  1.49104282e-01, ...,
          5.55727720e-01, -1.22244395e-01,  2.77456880e-01],
        [ 3.09472322e-01, -9.04355764e-01, -4.73911315e-01, ...,
          4.63262528e-01,  5.90811551e

In [26]:
# Predict expression on the learned ln(TPM+1) scale.
res3 = extract_feature(
    model=model,
    expr_array=input_df.values,
    output_feature_type='expr_level',
    batch_size=batch_size,
    amp_dtype=amp_dtype)


In [27]:
res3.shape


(4, 19973)

In [28]:
res3


array([[-2.609375  , -1.40625   ,  3.640625  , ...,  0.70703125,
         2.890625  ,  2.84375   ],
       [-2.640625  ,  1.75      ,  3.59375   , ...,  0.8125    ,
         2.9375    ,  3.0625    ],
       [-2.5       ,  3.71875   ,  3.421875  , ...,  0.8125    ,
         3.09375   ,  2.921875  ],
       [-2.765625  ,  0.62109375,  2.703125  , ...,  0.71875   ,
         3.125     ,  3.125     ]], dtype=float32)

In [29]:
# Save features with their sample and gene identifiers.
os.makedirs(output_path, exist_ok=True)
pd.DataFrame(res1, index=input_df.index).to_csv(output_path + '/sample_features.csv')
pd.DataFrame(res3, index=input_df.index, columns=bulkformer_gene_list).to_csv(
    output_path + '/expression_predictions.csv')
np.savez_compressed(
    output_path + '/selected_gene_features.npz',
    features=res2,
    sample_ids=input_df.index.to_numpy(dtype=str),
    gene_ids=np.array(interested_gene_list))


In [30]:
sorted(os.listdir(output_path))


['expression_predictions.csv',
 'sample_features.csv',
 'selected_gene_features.npz']